# Export YOLOX-Tiny to ONNX and check compatibility

Run cells in order after notebook 09. No retraining or dataset preparation occurs. Use the same Python kernel as the previous notebooks.

The current mobile app is not in this repository, so this notebook prepares and verifies a model package plus a reference adapter. It does not claim the mobile app can load the new model without changes.

This is a new native YOLOX model, not an overwrite of yolo11n-retrained.onnx. Input: float32 [1,3,416,416], BGR 0..255, top-left letterbox with padding 114. Output: decoded float32 [1,3549,7], columns cx,cy,w,h,objectness,p_licence,p_passport. Grid/stride decoding and probability activation are INCLUDED in the exported graph. NMS is NOT in the graph.

The calibrated threshold and class mapping are embedded in ONNX metadata and written to a JSON sidecar. Confidence is objectness times the highest class probability; class-aware NMS IoU is 0.65. The Python adapter preserves the earlier detect_document result keys, including null confidence/box for none.

The candidate is promoted to models/yolox-tiny-retrained.onnx only after CPU ONNX Runtime matches PyTorch outputs and the adapter matches the official YOLOX postprocessor on validation samples and uploaded phone photos. This is an export-equivalence check, not a new accuracy benchmark or mobile performance test. Percentages, elapsed time, ETA, and local finish time are printed for export and parity phases.

In [1]:
from pathlib import Path
import importlib.util
import shlex
import sys

PROJECT = Path("/home/joe/PycharmProjects/id-capture")
SOURCE = PROJECT / "training/vendor/YOLOX"
assert (SOURCE / "yolox").is_dir(), "Run notebook 07 first."
for directory in (str(PROJECT), str(SOURCE)):
    if directory not in sys.path:
        sys.path.insert(0, directory)

missing = [name for name in ("onnx", "onnxruntime")
           if importlib.util.find_spec(name) is None]
if missing:
    print("Run in a terminal:", shlex.join([sys.executable, "-m", "pip", "install", *missing]))
    raise RuntimeError("Install the missing export packages, restart the kernel, and rerun.")
print("Export packages are available. This notebook uses CPU; it does not start training.")

Export packages are available. This notebook uses CPU; it does not start training.


## Export a candidate

This uses the best checkpoint and saved calibration from notebooks 08/09. The fixed graph contains box decoding. The legacy PyTorch exporter is selected explicitly to avoid requiring onnxscript; no simplifier or Ultralytics exporter is used. Warnings about fixed trace sizes are expected because this model has a fixed input shape.

In [2]:
from training.yolox_export import export_candidate, validate_and_publish

bundle = export_candidate(PROJECT)
print("Candidate:", bundle["candidate"])
print("Input:", bundle["contract"]["input_shape"])
print("Output:", bundle["contract"]["output_shape"])
print("Confidence threshold:", bundle["contract"]["confidence"])

Loading checkpoint | 0.0% | Elapsed 0:00:00 | Remaining estimating | Estimated finish estimating
Loaded pretrained tensors: 462 | New class-head tensors: 0
Exporting ONNX | 33.3% | Elapsed 0:00:03 | Remaining 0:00:06 | Estimated finish 2026-10-21 19:15:23 BST
Candidate exported; parity checks are next | 100.0% | Elapsed 0:00:03 | Remaining 0:00:00 | Finished 2026-10-21 19:15:17 BST
Candidate: /home/joe/PycharmProjects/id-capture/models/yolox-tiny-retrained.candidate.onnx
Input: [1, 3, 416, 416]
Output: [1, 3549, 7]
Confidence threshold: 0.2


## Validate PyTorch / ONNX / adapter parity and publish locally

The next cell samples validation images by source/class and all uploaded phone photos. It checks raw box/probability tensors, post-NMS detections, original-image coordinate mapping, and final document types. Any mismatch stops publication and leaves the candidate for inspection. Original YOLO11 files are preserved. Runtime on the actual phone still needs to be verified in the mobile app.

In [3]:
report = validate_and_publish(
    bundle,
    phone_root=PROJECT / "training/data/raw",
)
print("PASS:", report["images"], "images checked.")
print("Deployable model:", bundle["final"])
print("Read models/YOLOX_INTEGRATION.md before changing the mobile loader.")

ONNX parity 1/33 | 3.0% | Elapsed 0:00:00 | Remaining 0:00:03 | Estimated finish 2026-10-21 19:15:27 BST
ONNX parity 2/33 | 6.1% | Elapsed 0:00:00 | Remaining 0:00:03 | Estimated finish 2026-10-21 19:15:27 BST
ONNX parity 3/33 | 9.1% | Elapsed 0:00:00 | Remaining 0:00:03 | Estimated finish 2026-10-21 19:15:27 BST
ONNX parity 4/33 | 12.1% | Elapsed 0:00:00 | Remaining 0:00:02 | Estimated finish 2026-10-21 19:15:27 BST
ONNX parity 5/33 | 15.2% | Elapsed 0:00:00 | Remaining 0:00:02 | Estimated finish 2026-10-21 19:15:27 BST
ONNX parity 6/33 | 18.2% | Elapsed 0:00:00 | Remaining 0:00:02 | Estimated finish 2026-10-21 19:15:27 BST
ONNX parity 7/33 | 21.2% | Elapsed 0:00:00 | Remaining 0:00:02 | Estimated finish 2026-10-21 19:15:26 BST
ONNX parity 8/33 | 24.2% | Elapsed 0:00:00 | Remaining 0:00:02 | Estimated finish 2026-10-21 19:15:26 BST
ONNX parity 9/33 | 27.3% | Elapsed 0:00:00 | Remaining 0:00:02 | Estimated finish 2026-10-21 19:15:26 BST
ONNX parity 10/33 | 30.3% | Elapsed 0:00:00 | Rem

## Reference Python integration

The runtime adapter requires ONNX Runtime, NumPy, OpenCV, and Pillow. It does not import PyTorch, YOLOX source, or Ultralytics. detect_document reads an image file with EXIF orientation applied; detect_bgr accepts an already upright HWC BGR uint8 frame. Returned boxes refer to upright original-image pixels. An RGB phone camera frame must be converted to BGR before using the equivalent preprocessing in the mobile app.

In [5]:
from id_capture.onnx_detector import OnnxDocumentDetector

detector = OnnxDocumentDetector(PROJECT / "models/yolox-tiny-retrained.onnx")
sample_path = next(
    Path(row["path"]) for row in report["rows"]
    if row["group"].startswith("phone:")
) if any(row["group"].startswith("phone:") for row in report["rows"]) else Path(report["rows"][0]["path"])
print(sample_path.name)
print(detector.detect_document(sample_path))

driving_licence1.jpg
{'document_type': 'driving_licence', 'detection_confidence': 0.9781813621520996, 'box_xyxy': [92.54608154296875, 475.91827392578125, 656.4304809570312, 886.342041015625]}
